# Exp3a Training Reproducibility Notebook (Release)

This notebook is the training-level reproducibility entry for the paper release.

Goals:
1. Train `MSE-only`, `DGSEA-only`, and `Hybrid(best)` from scratch.
2. Reconstruct core test table and paired-bootstrap confidence intervals.
3. Export publishable tables for conclusion-consistency checks.

Notes:
- This notebook is adapted from the original experiment notebook and de-duplicated for release.
- On CPU-only machines, runtime can be long. Use environment overrides for smaller pilot runs.


In [ ]:
# ===== 0) CONFIG (release-friendly defaults, env-overridable) =====
import os, random, math
from pathlib import Path

def _resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "dgsea_backend.py").exists():
            return cand
    return here

PROJECT_ROOT = _resolve_repo_root()

# ---------- paths ----------
DATA_PATH = os.environ.get("L1000_PARQUET", str(PROJECT_ROOT / "data" / "raw" / "smiles_signatures.parquet"))
# If CHEMBERTA_DIR points to local folder, it will be used.
# Otherwise this can be a HuggingFace model id (online download at first run).
MODEL_DIR = os.environ.get("CHEMBERTA_DIR", "ChemBERTa")
_OUT_DEFAULT = PROJECT_ROOT / "runs" / "chemberta_exp3a"
OUT_DIR = Path(os.environ.get("OUT_DIR", str(_OUT_DEFAULT))).expanduser().resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- experiment setup ----------
LOCAL_FILES_ONLY = os.environ.get("LOCAL_FILES_ONLY", "1") == "1"

SEED = int(os.environ.get("SEED", "42"))
DEVICE = os.environ.get("DEVICE", "auto")  # "auto" | "cuda" | "cpu"

# data split
TEST_FRAC = float(os.environ.get("TEST_FRAC", "0.15"))
VAL_FRAC  = float(os.environ.get("VAL_FRAC", "0.10"))

# optional dataset downsample for CPU pilot runs (0 = disabled)
MAX_SAMPLES = int(os.environ.get("MAX_SAMPLES", "0"))

# model/data
MAX_LEN = int(os.environ.get("MAX_LEN", "128"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "0"))
USE_FP = os.environ.get("USE_FP", "0") == "1"
FP_BITS = int(os.environ.get("FP_BITS", "2048"))
FP_RADIUS = int(os.environ.get("FP_RADIUS", "2"))
FILTER_INVALID_SMILES = os.environ.get("FILTER_INVALID_SMILES", "1") == "1"

# training
EPOCHS_MSE   = int(os.environ.get("EPOCHS_MSE", "5"))
EPOCHS_DGSEA = int(os.environ.get("EPOCHS_DGSEA", "5"))
# used in cell 17 (hybrid stage)
EPOCHS_HYBRID = int(os.environ.get("EPOCHS_HYBRID", "4"))
LR = float(os.environ.get("LR", "1e-3"))
WEIGHT_DECAY = float(os.environ.get("WEIGHT_DECAY", "1e-4"))
GRAD_CLIP = float(os.environ.get("GRAD_CLIP", "1.0"))

# pathway setup
PATHWAY_KEY = os.environ.get("PATHWAY_KEY", "P53_pathway")

# dGSEA kwargs
DGSEA_KW = dict(tau_rank=0.80, tau_prefix=1.10, tau_abs=0.70, p=1.0)

print("PROJECT_ROOT:", str(PROJECT_ROOT))
print("DATA_PATH:", DATA_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("OUT_DIR  :", str(OUT_DIR))
print("PATHWAY_KEY:", PATHWAY_KEY)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)
print("EPOCHS(MSE/DGSEA/HYB):", EPOCHS_MSE, EPOCHS_DGSEA, EPOCHS_HYBRID)


In [ ]:
# ===== 1) IMPORTS / SEED / DEVICE / QUIET LOGS =====
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)

device = torch.device("cuda" if (DEVICE == "cuda" and torch.cuda.is_available()) else "cpu")
print("device:", device)

# quiet RDKit / transformers logs (avoid IOPub spam)
try:
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.*")
except Exception as e:
    print("[WARN] RDKit not available or cannot mute logs:", e)

try:
    import transformers
    transformers.utils.logging.set_verbosity_error()
except Exception as e:
    print("[WARN] transformers logging not set:", e)


In [ ]:
# ===== release helper) resolve auto device =====
if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device("cuda" if (DEVICE == "cuda" and torch.cuda.is_available()) else "cpu")
print("[release] resolved device:", device)


In [ ]:
# ===== 2) LOAD DATA (parquet) + DETECT SMILES / GENE COLS =====
from pathlib import Path
assert Path(DATA_PATH).exists(), f"DATA_PATH not found: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH)
print("raw df:", df.shape)

# detect smiles column
smiles_candidates = ["canonical_smiles", "smiles", "SMILES", "canonical", "canon_smiles"]
smiles_col = None
for c in smiles_candidates:
    if c in df.columns:
        smiles_col = c
        break
if smiles_col is None:
    # heuristic: first object column with typical SMILES chars
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    for c in obj_cols:
        sample = df[c].dropna().astype(str).head(50).tolist()
        if len(sample) and any(("C" in s or "N" in s or "=" in s or "#" in s) for s in sample):
            smiles_col = c
            break
assert smiles_col is not None, "Cannot detect SMILES column."

# gene columns
gene_cols = [c for c in df.columns if c.startswith("gene_")]
assert len(gene_cols) > 0, "No gene_ columns found."

# basic clean: drop NA smiles, keep finite Y
df = df.dropna(subset=[smiles_col]).reset_index(drop=True)
Y = df[gene_cols].to_numpy(dtype=np.float32)
mask_finite = np.isfinite(Y).all(axis=1)
df = df.loc[mask_finite].reset_index(drop=True)
Y = df[gene_cols].to_numpy(dtype=np.float32)

print("smiles_col:", smiles_col)
print("#genes    :", len(gene_cols))
print("after finite:", df.shape)

# optional: filter invalid smiles (RDKit parseable)
if FILTER_INVALID_SMILES:
    try:
        from rdkit import Chem
        ok = []
        for s in df[smiles_col].astype(str).tolist():
            m = Chem.MolFromSmiles(s)
            ok.append(m is not None)
        ok = np.array(ok, dtype=bool)
        before = len(df)
        df = df.loc[ok].reset_index(drop=True)
        Y = df[gene_cols].to_numpy(dtype=np.float32)
        print(f"valid smiles filter: {before} -> {len(df)}")
    except Exception as e:
        print("[WARN] cannot filter invalid smiles:", e)

G = len(gene_cols)


In [ ]:
# ===== release helper) optional downsample for CPU pilot runs =====
if MAX_SAMPLES > 0 and len(df) > MAX_SAMPLES:
    rng_keep = np.random.RandomState(SEED)
    keep_idx = np.sort(rng_keep.choice(len(df), size=MAX_SAMPLES, replace=False))
    df = df.iloc[keep_idx].reset_index(drop=True)
    Y = df[gene_cols].to_numpy(dtype=np.float32)
    print(f"[release] downsample applied: N={len(df)}")
else:
    print(f"[release] downsample disabled; N={len(df)}")


In [ ]:
# ===== 3) SPLIT (train/val/test) =====
N = len(df)
perm = np.random.RandomState(SEED).permutation(N)

n_test = int(round(TEST_FRAC * N))
n_val  = int(round(VAL_FRAC  * N))
n_train = N - n_val - n_test

idx_train = perm[:n_train]
idx_val   = perm[n_train:n_train+n_val]
idx_test  = perm[n_train+n_val:]

print("N:", N, "| train:", len(idx_train), "val:", len(idx_val), "test:", len(idx_test))


In [ ]:
# ===== 4) LABEL STANDARDIZE (train stats) =====
y_mu = Y[idx_train].mean(axis=0, keepdims=True).astype(np.float32)   # [1,G]
y_std = Y[idx_train].std(axis=0, keepdims=True).astype(np.float32)   # [1,G]
y_std = np.maximum(y_std, 1e-6)

Yz = (Y - y_mu) / y_std

# torch buffers for inverse
y_mu_t  = torch.from_numpy(y_mu).to(device)   # [1,G]
y_std_t = torch.from_numpy(y_std).to(device)  # [1,G]

def inverse_standardize(yz: torch.Tensor) -> torch.Tensor:
    # yz: [B,G]
    return yz * y_std_t + y_mu_t

print("Y mean/std (train) sanity:", float(Y[idx_train].mean()), float(Y[idx_train].std()))


In [ ]:
# ===== 5) PATHWAYS (hard-coded) + GENE MAPPING =====
# ---- Five small pathways (directional) ----
PATHWAYS = {
    "P53_pathway": {
        "pretty": "p53 tumor suppressor",
        "up":  ["CDKN1A","GADD45A","GADD45B","PMAIP1","TP53BP1","TP53BP2"],
        "down":["BCL2"],
        "desc":"DNA损伤/p53激活；6上调+1下调"
    },
    "CellCycle_CDK": {
        "pretty":"Cell cycle core (CDK/Cyclins)",
        "up":  ["CDKN1A"],
        "down":["CDK1","CDK2","CCNA2","CCNB2"],
        "desc":"G1/S, G2/M 关键因子；抗增殖/周期阻滞"
    },
    "HeatShock_DNAJ": {
        "pretty":"Heat shock response (DNAJ/HSP40)",
        "up":["DNAJB1","DNAJB2","DNAJB6","DNAJA3"],
        "down":[],
        "desc":"蛋白毒性压力；HSP90/HSP70体系"
    },
    "Ubiquitin_DUB": {
        "pretty":"Ubiquitin-proteasome (DUBs)",
        "up":["USP1","USP7","USP14","USP22"],
        "down":[],
        "desc":"去泛素化酶；蛋白稳态/降解"
    },
    "DNA_damage_GADD45": {
        "pretty":"DNA damage (GADD45 family)",
        "up":["GADD45A","GADD45B"],
        "down":[],
        "desc":"最小DNA损伤标记集"
    }
}

# ---- minimal symbol->Entrez mapping (human) for the genes above ----
# 来源：NCBI Gene（Gene ID）
SYMBOL2ENTREZ = {
    "CDKN1A": 1026,
    "GADD45A": 1647,
    "GADD45B": 4616,
    "PMAIP1": 5366,
    "TP53BP1": 7158,
    "TP53BP2": 7159,
    "BCL2": 596,
    "CDK1": 983,
    "CDK2": 1017,
    "CCNA2": 890,
    "CCNB2": 9133,
    "DNAJB1": 3337,
    "DNAJB2": 3300,
    "DNAJB6": 10049,
    "DNAJA3": 9093,
    "USP1": 7398,
    "USP7": 7874,
    "USP14": 9097,
    "USP22": 23326,
}

def sym2id(symbols):
    ids = []
    missing = []
    for s in symbols:
        s = str(s).strip()
        if s in SYMBOL2ENTREZ:
            ids.append(int(SYMBOL2ENTREZ[s]))
        else:
            missing.append(s)
    return ids, missing

# ---- derive gene identifiers from columns ----
gene_ids_raw = []
for c in gene_cols:
    if c.startswith("gene_"):
        gene_ids_raw.append(c[5:])
    else:
        gene_ids_raw.append(c)

# heuristic: decide if gene ids look like Entrez IDs (digits)
is_digit = [gid.isdigit() for gid in gene_ids_raw]
digit_ratio = float(np.mean(is_digit))
GENE_ID_KIND = "entrez" if digit_ratio > 0.8 else "symbol"
print("GENE_ID_KIND:", GENE_ID_KIND, "| digit_ratio:", digit_ratio)

# build mapping to column index
gene2idx = {}
for i,gid in enumerate(gene_ids_raw):
    gene2idx[gid] = i
    gene2idx[gid.upper()] = i
    gene2idx[gene_cols[i]] = i
    gene2idx[gene_cols[i].upper()] = i

entrez2idx = {}
if GENE_ID_KIND == "entrez":
    for i,gid in enumerate(gene_ids_raw):
        if gid.isdigit():
            entrez2idx[int(gid)] = i

def genes_to_indices(genes):
    """genes: list of gene symbols (preferred) or raw ids matching columns."""
    idxs = []
    missing = []
    for g in genes:
        g = str(g).strip()
        hit = None
        # try direct match (symbol-style)
        if g in gene2idx:
            hit = gene2idx[g]
        elif g.upper() in gene2idx:
            hit = gene2idx[g.upper()]
        # if not found and dataset uses entrez ids, map symbol->entrez->idx
        if hit is None and GENE_ID_KIND == "entrez":
            if g in SYMBOL2ENTREZ:
                eid = int(SYMBOL2ENTREZ[g])
                if eid in entrez2idx:
                    hit = entrez2idx[eid]
        if hit is None:
            missing.append(g)
        else:
            idxs.append(int(hit))
    idxs = sorted(set(idxs))
    return idxs, missing

def indices_to_mask(idxs, G):
    m = torch.zeros((G,), dtype=torch.bool)
    if len(idxs) > 0:
        m[idxs] = True
    return m

# build masks
path_rows = []
PATH_MASKS = {}  # key -> dict(up_mask, dn_mask, ...)
for key,cfg in PATHWAYS.items():
    up = cfg.get("up", [])
    dn = cfg.get("down", [])
    up_idx, miss_u = genes_to_indices(up)
    dn_idx, miss_d = genes_to_indices(dn)
    cfg2 = dict(cfg)
    cfg2["up_idx"] = up_idx
    cfg2["dn_idx"] = dn_idx
    cfg2["missing"] = {"up": miss_u, "down": miss_d}
    cfg2["up_mask"] = indices_to_mask(up_idx, G)
    cfg2["dn_mask"] = indices_to_mask(dn_idx, G)
    PATH_MASKS[key] = cfg2
    path_rows.append({
        "pathway": key,
        "pretty": cfg["pretty"],
        "n_up": len(up_idx),
        "n_down": len(dn_idx),
        "missing_up": miss_u,
        "missing_down": miss_d,
        "desc": cfg["desc"],
    })

pd.DataFrame(path_rows)


In [ ]:
# ===== release helper) ensure src import path =====
import sys
_SRC_CANDS = [Path.cwd() / 'src', Path.cwd().parent / 'src']
_SRC = None
for _p in _SRC_CANDS:
    if (_p / 'dgsea_backend.py').exists():
        _SRC = _p.resolve()
        break
if _SRC is None:
    raise FileNotFoundError('Cannot locate src/dgsea_backend.py from current working directory.')
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
print('[release] using src path:', _SRC)


In [ ]:
# ===== 6) IMPORT dgsea_backend + robust wrappers (keyword-compatible) =====
# 目标：兼容不同 dgsea_backend 版本的参数名（例如不接受 tau，只接受 tau_abs/tau_rank 等）
import inspect

try:
    import dgsea_backend as DG
except Exception as e:
    raise RuntimeError(
        "Cannot import dgsea_backend. "
        "请确认你当前 kernel 环境里已经安装/可见 dgsea_backend。原始报错：\n" + str(e)
    )

# ---- 1) Introspect dgsea_des signature (if possible) ----
try:
    _DGSEA_SIG = inspect.signature(DG.dgsea_des)
    _DGSEA_PARAMS = set(_DGSEA_SIG.parameters.keys())
    print("[dgsea] DG.dgsea_des signature:", _DGSEA_SIG)
except Exception as e:
    _DGSEA_SIG = None
    _DGSEA_PARAMS = None
    print("[dgsea][warn] cannot inspect signature of DG.dgsea_des; will pass no kwargs. err:", repr(e))

def _pick_first(existing: set, candidates):
    for c in candidates:
        if c in existing:
            return c
    return None

def _adapt_dgsea_kw(user_kw: dict):
    """
    将用户的 DGSEA_KW 映射为 dgsea_backend.dgsea_des 实际接受的关键字。
    - 若无法 introspect，则返回空 dict（使用 dgsea_backend 默认参数）
    - 支持常见别名：tau -> tau_abs/tauabs/...
    """
    if not isinstance(user_kw, dict):
        return {}
    if _DGSEA_PARAMS is None:
        return {}

    out = {}

    # 1) 直接保留后端支持的 key
    for k, v in user_kw.items():
        if k in _DGSEA_PARAMS:
            out[k] = v

    # 2) 常见别名映射（只在目标参数存在且尚未设置时写入）
    def _set_if_missing(dst_key, v):
        if dst_key is not None and dst_key not in out:
            out[dst_key] = v

    # tau（常被用作 softmax/abs 温度）
    if "tau" in user_kw and "tau" not in _DGSEA_PARAMS:
        v = user_kw["tau"]
        dst = _pick_first(_DGSEA_PARAMS, ["tau_abs", "tauabs", "tauA", "tau_abs_", "tau_softmax", "tau_agg", "tau_es", "tauAbs", "t_abs", "ta"])
        _set_if_missing(dst, v)

    # tau_rank
    if "tau_rank" in user_kw and "tau_rank" not in _DGSEA_PARAMS:
        v = user_kw["tau_rank"]
        dst = _pick_first(_DGSEA_PARAMS, ["tau_rank", "taurank", "tau_r", "taur", "tauRank", "tr"])
        _set_if_missing(dst, v)

    # tau_prefix
    if "tau_prefix" in user_kw and "tau_prefix" not in _DGSEA_PARAMS:
        v = user_kw["tau_prefix"]
        dst = _pick_first(_DGSEA_PARAMS, ["tau_prefix", "tauprefix", "tau_p", "taup", "tauPrefix", "tp"])
        _set_if_missing(dst, v)

    # eps
    if "eps" in user_kw and "eps" not in _DGSEA_PARAMS:
        v = user_kw["eps"]
        dst = _pick_first(_DGSEA_PARAMS, ["eps", "epsilon", "eps_", "EPS"])
        _set_if_missing(dst, v)

    # p / power
    if "p" in user_kw and "p" not in _DGSEA_PARAMS:
        v = user_kw["p"]
        dst = _pick_first(_DGSEA_PARAMS, ["p", "power", "p_val", "p_"])
        _set_if_missing(dst, v)

    return out

DGSEA_KW_RAW = globals().get("DGSEA_KW", {})
DGSEA_KW_USE = _adapt_dgsea_kw(DGSEA_KW_RAW)

print("[dgsea] DGSEA_KW raw :", DGSEA_KW_RAW)
print("[dgsea] DGSEA_KW used:", DGSEA_KW_USE)

# ---- 2) unwrap helper ----
def dgsea_unwrap(out):
    """dgsea_backend 可能返回 Tensor 或 (Tensor, aux...)；递归取第一个 Tensor。"""
    if torch.is_tensor(out):
        return out
    if isinstance(out, (tuple, list)) and len(out) > 0:
        return dgsea_unwrap(out[0])
    raise TypeError(f"dgsea_backend.dgsea_des returned unsupported type: {type(out)}")

# ---- 3) public wrapper used by training/eval ----
def dgsea_des(scores: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    scores: [B,G]
    mask  : [G] bool
    returns: [B]
    """
    out = DG.dgsea_des(scores, mask, **DGSEA_KW_USE) if len(DGSEA_KW_USE) > 0 else DG.dgsea_des(scores, mask)
    out = dgsea_unwrap(out)

    # normalize shape to [B]
    if out.ndim == 0:
        out = out.view(1)
    if out.ndim == 2 and out.shape[1] == 1:
        out = out[:, 0]
    return out

# ---- 4) quick sanity checks (forward + autograd) ----
@torch.no_grad()
def dgsea_test_once():
    x = torch.randn(2, G, device=device)
    m = PATH_MASKS[PATHWAY_KEY]["up_mask"].to(device)
    o = dgsea_des(x, m)
    assert torch.is_tensor(o), "dgsea_des did not return torch.Tensor."
    print("[dgsea] forward OK. output shape:", tuple(o.shape), "| mean:", float(o.mean().item()))

def dgsea_autograd_check():
    x = torch.randn(2, G, device=device, requires_grad=True)
    m = PATH_MASKS[PATHWAY_KEY]["up_mask"].to(device)
    o = dgsea_des(x, m).sum()
    o.backward()
    assert x.grad is not None, "No grad returned."
    assert torch.isfinite(x.grad).all(), "Grad has NaN/Inf."
    print("[dgsea] autograd OK. grad mean abs:", float(x.grad.abs().mean().item()))

dgsea_test_once()
dgsea_autograd_check()

# ---- 5) RS helper (up - down) ----
def pathway_rs(scores: torch.Tensor, up_mask: torch.Tensor, dn_mask: torch.Tensor) -> torch.Tensor:
    """Return reversal score RS = ES(up) - ES(down). If no down genes, RS=ES(up)."""
    es_up = dgsea_des(scores, up_mask)
    if dn_mask is None or int(dn_mask.sum().item()) == 0:
        return es_up
    es_dn = dgsea_des(scores, dn_mask)
    return es_up - es_dn


In [ ]:
# ===== 7) COMPUTE empirical-calibrated dNES scaling (train true profiles) =====
from tqdm.auto import tqdm

up_mask = PATH_MASKS[PATHWAY_KEY]["up_mask"].to(device)
dn_mask = PATH_MASKS[PATHWAY_KEY]["dn_mask"].to(device)

def compute_true_rs(idx_list, batch_size=256):
    rs_all = []
    for s in range(0, len(idx_list), batch_size):
        batch_idx = idx_list[s:s+batch_size]
        y = torch.from_numpy(Y[batch_idx]).to(device)  # original space
        rs = pathway_rs(y, up_mask, dn_mask).detach().float().cpu().numpy()
        rs_all.append(rs)
    return np.concatenate(rs_all, axis=0)

rs_train_true = compute_true_rs(idx_train, batch_size=512)

# sign-specific scale (mean |RS| within each sign) — empirical calibrated dNES
pos = np.abs(rs_train_true[rs_train_true >= 0])
neg = np.abs(rs_train_true[rs_train_true < 0])

pos_scale = float(pos.mean()) if len(pos) else float(np.abs(rs_train_true).mean())
neg_scale = float(neg.mean()) if len(neg) else float(np.abs(rs_train_true).mean())
pos_scale = max(pos_scale, 1e-6)
neg_scale = max(neg_scale, 1e-6)

def to_dnes(rs: torch.Tensor) -> torch.Tensor:
    # rs: [B]
    denom = torch.where(rs >= 0, torch.tensor(pos_scale, device=rs.device), torch.tensor(neg_scale, device=rs.device))
    return rs / denom

print("empirical dNES scales | pos_scale=%.6f  neg_scale=%.6f" % (pos_scale, neg_scale))
print("train true RS stats: mean=%.4f std=%.4f p95=%.4f" % (rs_train_true.mean(), rs_train_true.std(), np.quantile(rs_train_true, 0.95)))


In [ ]:
# ===== 8) TOKENIZER / FP / DATASET / DATALOADER =====
from transformers import RobertaTokenizerFast

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR, local_files_only=LOCAL_FILES_ONLY)

# Morgan FP with new API (avoid deprecation warning)
_fp_cache = {}

def morgan_fp_cached(smiles: str, nbits=2048, radius=2):
    if smiles in _fp_cache:
        return _fp_cache[smiles]
    try:
        from rdkit import Chem
        from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            arr = np.zeros((nbits,), dtype=np.float32)
        else:
            gen = GetMorganGenerator(radius=radius, fpSize=nbits)
            fp = gen.GetFingerprint(mol)
            arr = np.zeros((nbits,), dtype=np.float32)
            # Convert to numpy
            from rdkit import DataStructs
            DataStructs.ConvertToNumpyArray(fp, arr)
    except Exception:
        arr = np.zeros((nbits,), dtype=np.float32)
    _fp_cache[smiles] = arr
    return arr

class L1000SmilesDataset(Dataset):
    def __init__(self, df, idx, smiles_col, Yz):
        self.df = df
        self.idx = np.asarray(idx)
        self.smiles_col = smiles_col
        self.Yz = Yz

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = int(self.idx[i])
        smi = str(self.df.iloc[j][self.smiles_col])
        yz  = self.Yz[j].astype(np.float32)  # [G]
        return smi, yz

def collate_fn(batch):
    smiles = [x[0] for x in batch]
    yz = torch.from_numpy(np.stack([x[1] for x in batch], axis=0))  # [B,G]
    tok = tokenizer(smiles, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    if USE_FP:
        fp = torch.from_numpy(np.stack([morgan_fp_cached(s, FP_BITS, FP_RADIUS) for s in smiles], axis=0))
    else:
        fp = None
    return {
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "fp": fp,
        "y_z": yz,
    }

train_ds = L1000SmilesDataset(df, idx_train, smiles_col, Yz)
val_ds   = L1000SmilesDataset(df, idx_val,   smiles_col, Yz)
test_ds  = L1000SmilesDataset(df, idx_test,  smiles_col, Yz)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print("loaders:", len(train_loader), len(val_loader), len(test_loader))


In [ ]:
# ===== 9) MODEL (encoder frozen) =====
from transformers import RobertaModel

class ChemBERTaRegressor(nn.Module):
    def __init__(self, model_dir: str, out_dim: int, use_fp: bool, fp_dim: int):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_dir, local_files_only=LOCAL_FILES_ONLY)
        self.hidden = int(self.encoder.config.hidden_size)
        self.use_fp = bool(use_fp)

        # freeze encoder
        for p in self.encoder.parameters():
            p.requires_grad = False

        if self.use_fp:
            self.fp_mlp = nn.Sequential(
                nn.Linear(fp_dim, self.hidden),
                nn.ReLU(),
                nn.Linear(self.hidden, self.hidden),
            )
            head_in = self.hidden * 2
        else:
            self.fp_mlp = None
            head_in = self.hidden

        self.head = nn.Sequential(
            nn.Linear(head_in, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, out_dim),
        )

    def forward(self, input_ids, attention_mask, fp=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [B,H]
        if self.use_fp:
            assert fp is not None
            fp = fp.to(cls.device).float()
            fp_feat = self.fp_mlp(fp)
            x = torch.cat([cls, fp_feat], dim=1)
        else:
            x = cls
        return self.head(x)  # [B,G]

def make_model():
    model = ChemBERTaRegressor(MODEL_DIR, out_dim=G, use_fp=USE_FP, fp_dim=FP_BITS)
    return model.to(device)

# sanity forward
b = next(iter(train_loader))
with torch.no_grad():
    m = make_model()
    y = m(b["input_ids"].to(device), b["attention_mask"].to(device), fp=(b["fp"].to(device) if b["fp"] is not None else None))
    print("pred shape:", tuple(y.shape))


In [ ]:
# ===== 10) METRICS =====
def rowwise_pearson(a: torch.Tensor, b: torch.Tensor, eps=1e-8) -> torch.Tensor:
    # a,b: [B,G]
    a = a - a.mean(dim=1, keepdim=True)
    b = b - b.mean(dim=1, keepdim=True)
    num = (a*b).sum(dim=1)
    den = torch.sqrt((a*a).sum(dim=1) + eps) * torch.sqrt((b*b).sum(dim=1) + eps)
    return num / (den + eps)

@torch.no_grad()
def eval_gene_metrics(model, loader, topk_list=(50,)):
    model.eval()
    r_all = []
    r_topk = {k: [] for k in topk_list}
    for batch in tqdm(loader, desc="eval-gene", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fp = batch["fp"].to(device) if batch["fp"] is not None else None
        y_z = batch["y_z"].to(device)

        pred_z = model(input_ids, attention_mask, fp=fp)
        pred_o = inverse_standardize(pred_z)
        true_o = inverse_standardize(y_z)

        r = rowwise_pearson(pred_o, true_o).detach().cpu()
        r_all.append(r)

        # top-k by |TRUE| per sample, compute correlation on those indices
        abs_true = true_o.abs()
        for k in topk_list:
            kk = min(int(k), true_o.shape[1])
            idx = torch.topk(abs_true, kk, dim=1).indices  # [B,kk]
            # gather per sample
            p = torch.gather(pred_o, 1, idx)
            t = torch.gather(true_o, 1, idx)
            rt = rowwise_pearson(p, t).detach().cpu()
            r_topk[k].append(rt)

    r_all = torch.cat(r_all, dim=0)
    out = {
        "ALL978_mean_r": float(r_all.mean().item()),
        "ALL978_median_r": float(r_all.median().item()),
    }
    for k in topk_list:
        v = torch.cat(r_topk[k], dim=0)
        out[f"Top{k}_mean_r"] = float(v.mean().item())
        out[f"Top{k}_median_r"] = float(v.median().item())
    return out

@torch.no_grad()
def eval_pathway_metrics(model, loader):
    model.eval()
    pred_list = []
    true_list = []
    for batch in tqdm(loader, desc="eval-path", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fp = batch["fp"].to(device) if batch["fp"] is not None else None
        y_z = batch["y_z"].to(device)

        pred_z = model(input_ids, attention_mask, fp=fp)
        pred_o = inverse_standardize(pred_z)
        true_o = inverse_standardize(y_z)

        rs_p = pathway_rs(pred_o, up_mask, dn_mask)
        rs_t = pathway_rs(true_o, up_mask, dn_mask)
        dnes_p = to_dnes(rs_p).detach().cpu().numpy()
        dnes_t = to_dnes(rs_t).detach().cpu().numpy()
        pred_list.append(dnes_p)
        true_list.append(dnes_t)

    pred = np.concatenate(pred_list)
    true = np.concatenate(true_list)

    # Pearson correlation across samples
    pred_c = pred - pred.mean()
    true_c = true - true.mean()
    denom = (np.sqrt((pred_c**2).sum()) * np.sqrt((true_c**2).sum()) + 1e-12)
    r = float((pred_c * true_c).sum() / denom)

    mse = float(np.mean((pred - true) ** 2))
    sign_acc = float(np.mean((pred >= 0) == (true >= 0)))

    return {"PATH_dNES_corr": r, "PATH_dNES_mse": mse, "PATH_sign_acc": sign_acc, "N": int(len(true))}



In [ ]:
# ===== 11) TRAINING =====
import time

def train_one(loss_mode: str, epochs: int, out_name: str):
    assert loss_mode in {"mse", "dgsea"}
    model = make_model()

    # trainable params: head (+ fp_mlp)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

    best = {"score": -1e18, "path": None, "epoch": -1}
    history = []

    for ep in range(1, epochs+1):
        model.train()
        t0 = time.time()
        losses = []

        for batch in tqdm(train_loader, desc=f"train-{loss_mode} e{ep}", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            fp = batch["fp"].to(device) if batch["fp"] is not None else None
            y_z = batch["y_z"].to(device)

            pred_z = model(input_ids, attention_mask, fp=fp)

            if loss_mode == "mse":
                loss = ((pred_z - y_z) ** 2).mean()
            else:
                # DGSEA-only: MSE on (empirical) dNES
                pred_o = inverse_standardize(pred_z)
                true_o = inverse_standardize(y_z)
                rs_p = pathway_rs(pred_o, up_mask, dn_mask)
                rs_t = pathway_rs(true_o, up_mask, dn_mask)
                dnes_p = to_dnes(rs_p)
                dnes_t = to_dnes(rs_t).detach()   # target not require grad
                loss = ((dnes_p - dnes_t) ** 2).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            opt.step()

            losses.append(float(loss.detach().cpu().item()))

        # evaluate on val (primary score = pathway corr, secondary = -mse)
        val_path = eval_pathway_metrics(model, val_loader)
        val_gene = eval_gene_metrics(model, val_loader, topk_list=(50,))

        score = float(val_path["PATH_dNES_corr"] - 0.05 * val_path["PATH_dNES_mse"])
        history.append({
            "epoch": ep,
            "train_loss": float(np.mean(losses)),
            **{f"val_{k}": v for k,v in val_path.items()},
            **{f"val_{k}": v for k,v in val_gene.items()},
            "score": score,
            "sec": time.time()-t0,
        })

        print(f"[{loss_mode}] ep={ep} train_loss={np.mean(losses):.4f} | "
              f"VAL PATH corr={val_path['PATH_dNES_corr']:.4f} mse={val_path['PATH_dNES_mse']:.4f} sign={val_path['PATH_sign_acc']:.3f} | "
              f"VAL ALL978={val_gene['ALL978_mean_r']:.4f} Top50={val_gene['Top50_mean_r']:.4f} | score={score:.4f}")

        if score > best["score"]:
            best["score"] = score
            best["epoch"] = ep
            ckpt_path = OUT_DIR / f"{out_name}_best.pt"
            torch.save({
                "model": model.state_dict(),
                "loss_mode": loss_mode,
                "epoch": ep,
                "score": score,
                "PATHWAY_KEY": PATHWAY_KEY,
                "DGSEA_KW": DGSEA_KW,
                "pos_scale": pos_scale,
                "neg_scale": neg_scale,
            }, ckpt_path)
            best["path"] = str(ckpt_path)

    return best, pd.DataFrame(history)

best_mse, hist_mse = train_one("mse", epochs=EPOCHS_MSE, out_name="exp3a_mse")
best_dg,  hist_dg  = train_one("dgsea", epochs=EPOCHS_DGSEA, out_name="exp3a_dgsea")

print("best_mse:", best_mse)
print("best_dg :", best_dg)


In [ ]:
# ===== 12) FINAL TEST EVAL + SUMMARY =====
def load_model(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = make_model()
    m.load_state_dict(ckpt["model"], strict=True)
    return m.to(device), ckpt

m_mse, ck_mse = load_model(best_mse["path"])
m_dg,  ck_dg  = load_model(best_dg["path"])

test_gene_mse = eval_gene_metrics(m_mse, test_loader, topk_list=(50,))
test_path_mse = eval_pathway_metrics(m_mse, test_loader)

test_gene_dg  = eval_gene_metrics(m_dg,  test_loader, topk_list=(50,))
test_path_dg  = eval_pathway_metrics(m_dg,  test_loader)

summary = pd.DataFrame([
    {"setting": "MSE-only",   **test_gene_mse, **test_path_mse},
    {"setting": "DGSEA-only", **test_gene_dg,  **test_path_dg},
]).set_index("setting")

display(summary)

print("\n===== DELTA (DGSEA-only - MSE-only) =====")
for k in ["PATH_dNES_corr","PATH_dNES_mse","PATH_sign_acc","ALL978_mean_r","Top50_mean_r"]:
    if k in summary.columns:
        dv = float(summary.loc["DGSEA-only", k] - summary.loc["MSE-only", k])
        print(f"{k}: {dv:+.6f}")


In [ ]:
# ===== (LOG CONTROL) Suppress HuggingFace tokenizers fork-parallelism warning + reduce spam =====
import os
import warnings

# 1) HuggingFace tokenizers: disable parallelism to avoid fork warnings/deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2) Reduce other common spam
warnings.filterwarnings("once")  # or "ignore" if you want absolutely silent

# transformers logging
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception as e:
    print("[warn] cannot set transformers logging:", repr(e))

# RDKit logging (optional)
try:
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.warning")
    RDLogger.DisableLog("rdApp.error")
except Exception:
    pass

print("[log] TOKENIZERS_PARALLELISM =", os.environ.get("TOKENIZERS_PARALLELISM"))
print("[log] logging config applied.")


In [ ]:
# ===== 14) EXP3a+: Multi-pathway dNES calibration (robust sign-specific denominators) =====
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- sanity: required globals from earlier cells ----
assert "PATH_MASKS" in globals() and isinstance(PATH_MASKS, dict) and len(PATH_MASKS) > 0
assert "PATHWAYS" in globals() and isinstance(PATHWAYS, dict) and len(PATHWAYS) > 0
assert "idx_train" in globals() and len(idx_train) > 0
assert "Y" in globals(), "Expect original-space gene matrix `Y` (numpy) to exist."
assert "G" in globals() and int(G) > 0
assert "device" in globals()
assert "pathway_rs" in globals(), "Expect `pathway_rs(scores, up_mask, dn_mask)` to exist."

# ---- choose pathways for exp3a (default: all 5 hard-coded) ----
PATHWAY_KEYS_EXP3A = list(PATHWAYS.keys())
print("[exp3a] PATHWAY_KEYS_EXP3A =", PATHWAY_KEYS_EXP3A)

# ---- robust estimator: trimmed mean + winsorized mean (sign-specific) ----
def robust_mean(x: np.ndarray, rho: float = 0.10, lam: float = 0.50) -> float:
    """
    x: 1D positive values (e.g., |RS| for positive-sign samples)
    rho: trimming proportion in (0, 0.5)
    lam: shrinkage coefficient in [0,1], combine trimmed and winsorized
    """
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("nan")
    x = np.sort(x)
    n = x.size
    k = int(np.floor(rho * n))
    if 2*k >= n:
        # too small; fallback to mean
        return float(x.mean())

    # trimmed mean
    x_trim = x[k:n-k]
    mu_trim = float(x_trim.mean())

    # winsorized mean
    lo = x[k]
    hi = x[n-k-1]
    x_win = x.copy()
    x_win[:k] = lo
    x_win[n-k:] = hi
    mu_win = float(x_win.mean())

    mu = (1.0 - lam) * mu_trim + lam * mu_win
    return float(mu)

# ---- compute true RS for a given pathway on TRAIN only (original gene space) ----
@torch.no_grad()
def compute_true_rs_for_key(key: str, idx_list, batch_size: int = 512) -> np.ndarray:
    up_mask = PATH_MASKS[key]["up_mask"].to(device)
    dn_mask = PATH_MASKS[key]["dn_mask"].to(device)
    rs_all = []
    for s in range(0, len(idx_list), batch_size):
        batch_idx = idx_list[s:s+batch_size]
        y = torch.from_numpy(Y[batch_idx]).to(device).float()  # [B,G], original space
        rs = pathway_rs(y, up_mask, dn_mask).detach().cpu().numpy().reshape(-1)
        rs_all.append(rs)
    return np.concatenate(rs_all, axis=0)

# ---- build per-pathway scales (pos/neg) ----
DNES_SCALES = {}  # key -> dict(pos_scale, neg_scale, stats...)
rows = []

RHO_TRIM = 0.10
LAMBDA_SHRINK = 0.50
EPS_SCALE = 1e-6

for key in PATHWAY_KEYS_EXP3A:
    rs_train = compute_true_rs_for_key(key, idx_train, batch_size=512)

    pos = np.abs(rs_train[rs_train > 0])
    neg = np.abs(rs_train[rs_train < 0])

    # robust sign-specific denominators
    pos_scale = robust_mean(pos, rho=RHO_TRIM, lam=LAMBDA_SHRINK)
    neg_scale = robust_mean(neg, rho=RHO_TRIM, lam=LAMBDA_SHRINK)

    # fallbacks if extremely small / empty
    fallback = float(np.mean(np.abs(rs_train))) if np.isfinite(rs_train).any() else 1.0
    if not np.isfinite(pos_scale): pos_scale = fallback
    if not np.isfinite(neg_scale): neg_scale = fallback

    pos_scale = max(float(pos_scale), EPS_SCALE)
    neg_scale = max(float(neg_scale), EPS_SCALE)

    DNES_SCALES[key] = {
        "pos_scale": pos_scale,
        "neg_scale": neg_scale,
        "rho": RHO_TRIM,
        "lam": LAMBDA_SHRINK,
        "rs_mean": float(rs_train.mean()),
        "rs_std": float(rs_train.std()),
        "rs_p95_abs": float(np.quantile(np.abs(rs_train), 0.95)),
        "n_pos": int((rs_train > 0).sum()),
        "n_neg": int((rs_train < 0).sum()),
    }

    rows.append({
        "pathway": key,
        "pretty": PATHWAYS[key]["pretty"],
        "n_up": int(PATH_MASKS[key]["up_mask"].sum().item()),
        "n_down": int(PATH_MASKS[key]["dn_mask"].sum().item()),
        "pos_scale": pos_scale,
        "neg_scale": neg_scale,
        "rs_mean": DNES_SCALES[key]["rs_mean"],
        "rs_std": DNES_SCALES[key]["rs_std"],
        "rs_p95_abs": DNES_SCALES[key]["rs_p95_abs"],
        "n_pos": DNES_SCALES[key]["n_pos"],
        "n_neg": DNES_SCALES[key]["n_neg"],
    })

df_scales = pd.DataFrame(rows).sort_values("pathway")
display(df_scales)

# ---- helpers: RS -> dNES (per key) ----
def to_dnes_key(rs: torch.Tensor, key: str) -> torch.Tensor:
    """
    rs: [B] tensor
    returns: [B] dNES using sign-specific denominators for that pathway
    """
    ps = float(DNES_SCALES[key]["pos_scale"])
    ns = float(DNES_SCALES[key]["neg_scale"])
    denom = torch.where(rs >= 0, torch.tensor(ps, device=rs.device), torch.tensor(ns, device=rs.device))
    return rs / denom

@torch.no_grad()
def dnes_from_scores_key(scores: torch.Tensor, key: str) -> torch.Tensor:
    """
    scores: [B,G] original space
    returns: dNES [B]
    """
    up_mask = PATH_MASKS[key]["up_mask"].to(scores.device)
    dn_mask = PATH_MASKS[key]["dn_mask"].to(scores.device)
    rs = pathway_rs(scores.float(), up_mask, dn_mask)
    return to_dnes_key(rs, key)

print("[exp3a] dNES scales ready. (robust sign-specific denominators)")


In [ ]:
# ===== 15) EXP3a+: Indexed dataloaders + precompute TRUE dNES for all pathways =====
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

assert "df" in globals()
assert "smiles_col" in globals()
assert "Y" in globals() and isinstance(Y, np.ndarray)
assert "Yz" in globals() and isinstance(Yz, np.ndarray)
assert "tokenizer" in globals()
assert "MAX_LEN" in globals()
assert "BATCH_SIZE" in globals()
assert "NUM_WORKERS" in globals()
assert "USE_FP" in globals()
assert "FP_BITS" in globals()
assert "FP_RADIUS" in globals()
assert "morgan_fp_cached" in globals()

class L1000SmilesDatasetIdx(Dataset):
    def __init__(self, df, idx, smiles_col, Yz):
        self.df = df
        self.idx = np.asarray(idx)
        self.smiles_col = smiles_col
        self.Yz = Yz

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = int(self.idx[i])
        smi = str(self.df.iloc[j][self.smiles_col])
        yz  = self.Yz[j].astype(np.float32)  # [G]
        return j, smi, yz

def collate_fn_idx(batch):
    idx = torch.tensor([x[0] for x in batch], dtype=torch.long)  # [B]
    smiles = [x[1] for x in batch]
    yz = torch.from_numpy(np.stack([x[2] for x in batch], axis=0))  # [B,G]

    tok = tokenizer(smiles, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    fp = None
    if USE_FP:
        fp = torch.from_numpy(np.stack([morgan_fp_cached(s, FP_BITS, FP_RADIUS) for s in smiles], axis=0))
    return {
        "idx": idx,
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "fp": fp,
        "y_z": yz,
    }

train_ds_idx = L1000SmilesDatasetIdx(df, idx_train, smiles_col, Yz)
val_ds_idx   = L1000SmilesDatasetIdx(df, idx_val,   smiles_col, Yz)
test_ds_idx  = L1000SmilesDatasetIdx(df, idx_test,  smiles_col, Yz)

train_loader_idx = DataLoader(train_ds_idx, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, collate_fn=collate_fn_idx)
val_loader_idx   = DataLoader(val_ds_idx,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn_idx)
test_loader_idx  = DataLoader(test_ds_idx,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn_idx)

print("[exp3a] indexed loaders ready:",
      "train/val/test =", len(train_ds_idx), len(val_ds_idx), len(test_ds_idx))

# ---- precompute TRUE dNES for all indices (union of train/val/test) for all pathways ----
IDX_ALL = np.unique(np.concatenate([np.asarray(idx_train), np.asarray(idx_val), np.asarray(idx_test)], axis=0))
print("[exp3a] precompute TRUE dNES on n =", len(IDX_ALL), "samples; pathways =", len(PATHWAY_KEYS_EXP3A))

DNES_TRUE_ALL = {k: np.full((len(df),), np.nan, dtype=np.float32) for k in PATHWAY_KEYS_EXP3A}

@torch.no_grad()
def _precompute_true_dnes(batch_idx: np.ndarray):
    y = torch.from_numpy(Y[batch_idx]).to(device).float()  # original space [B,G]
    out = {}
    for key in PATHWAY_KEYS_EXP3A:
        up_mask = PATH_MASKS[key]["up_mask"].to(device)
        dn_mask = PATH_MASKS[key]["dn_mask"].to(device)
        rs = pathway_rs(y, up_mask, dn_mask)  # [B]
        dnes = to_dnes_key(rs, key).detach().cpu().numpy().astype(np.float32)
        out[key] = dnes.reshape(-1)
    return out

BS_PRE = 512
for s in tqdm(range(0, len(IDX_ALL), BS_PRE), desc="precompute-true-dnes"):
    bidx = IDX_ALL[s:s+BS_PRE]
    pack = _precompute_true_dnes(bidx)
    for key in PATHWAY_KEYS_EXP3A:
        DNES_TRUE_ALL[key][bidx] = pack[key]

# sanity: no NaNs on split indices
def _check_nan(key, idx_list, name):
    arr = DNES_TRUE_ALL[key][np.asarray(idx_list)]
    frac_nan = float(np.mean(~np.isfinite(arr)))
    print(f"[exp3a] TRUE dNES nan fraction | {key} | {name}: {frac_nan:.6f}")
    assert frac_nan == 0.0, f"Found NaNs in TRUE dNES for {key} on {name} split."

for key in PATHWAY_KEYS_EXP3A:
    _check_nan(key, idx_train, "train")
    _check_nan(key, idx_val, "val")
    _check_nan(key, idx_test, "test")

print("[exp3a] TRUE dNES cache ready.")


In [ ]:
# ===== 16) EXP3a+: Multi-pathway evaluation utilities (arrays + summary tables) =====
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

assert "inverse_standardize" in globals()
assert "make_model" in globals()

def _pearson_np(a, b, eps=1e-12):
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    ac = a - a.mean()
    bc = b - b.mean()
    den = (np.sqrt((ac*ac).sum()) * np.sqrt((bc*bc).sum()) + eps)
    return float((ac*bc).sum() / den)

@torch.no_grad()
def collect_gene_arrays(model, loader_idx, topk=50):
    model.eval()
    r_all = []
    r_top = []

    for batch in tqdm(loader_idx, desc="collect-gene", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fp = batch["fp"]
        fp = fp.to(device) if fp is not None else None
        y_z = batch["y_z"].to(device)

        pred_z = model(input_ids, attention_mask, fp=fp)
        pred_o = inverse_standardize(pred_z).float()
        true_o = inverse_standardize(y_z).float()

        # all-978 rowwise Pearson
        ra = rowwise_pearson(pred_o, true_o).detach().cpu().numpy()
        r_all.append(ra)

        # top-k by |true|
        k = min(int(topk), true_o.shape[1])
        idx = torch.topk(true_o.abs(), k, dim=1).indices
        p = torch.gather(pred_o, 1, idx)
        t = torch.gather(true_o, 1, idx)
        rt = rowwise_pearson(p, t).detach().cpu().numpy()
        r_top.append(rt)

    r_all = np.concatenate(r_all, axis=0)
    r_top = np.concatenate(r_top, axis=0)
    return {"r_all": r_all, f"r_top{topk}": r_top}

@torch.no_grad()
def collect_pathway_arrays(model, loader_idx, keys):
    """
    returns dict:
      key -> dict(pred=np.ndarray [N], true=np.ndarray [N])
    """
    model.eval()
    pred_buf = {k: [] for k in keys}
    true_buf = {k: [] for k in keys}

    for batch in tqdm(loader_idx, desc="collect-path", leave=False):
        idx = batch["idx"].cpu().numpy().astype(np.int64)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fp = batch["fp"]
        fp = fp.to(device) if fp is not None else None
        y_z = batch["y_z"].to(device)

        pred_z = model(input_ids, attention_mask, fp=fp)
        pred_o = inverse_standardize(pred_z).float()

        # compute predicted dNES for each pathway
        for key in keys:
            up_mask = PATH_MASKS[key]["up_mask"].to(device)
            dn_mask = PATH_MASKS[key]["dn_mask"].to(device)
            rs = pathway_rs(pred_o, up_mask, dn_mask)
            dnes_p = to_dnes_key(rs, key).detach().cpu().numpy().astype(np.float32)
            pred_buf[key].append(dnes_p.reshape(-1))

            dnes_t = DNES_TRUE_ALL[key][idx].astype(np.float32)
            true_buf[key].append(dnes_t.reshape(-1))

    out = {}
    for key in keys:
        p = np.concatenate(pred_buf[key], axis=0)
        t = np.concatenate(true_buf[key], axis=0)
        out[key] = {"pred": p, "true": t}
    return out

def summarize_pathway_metrics(path_arrays: dict):
    """
    path_arrays: output of collect_pathway_arrays
    returns:
      macro dict + per-pathway DataFrame
    """
    rows = []
    for key, pack in path_arrays.items():
        p = pack["pred"]; t = pack["true"]
        r = _pearson_np(p, t)
        mse = float(np.mean((p - t)**2))
        sign_acc = float(np.mean((p >= 0) == (t >= 0)))
        rows.append({"pathway": key, "corr": r, "mse": mse, "sign_acc": sign_acc, "N": int(len(t))})
    df = pd.DataFrame(rows).sort_values("pathway").set_index("pathway")

    macro = {
        "PATH_macro_corr": float(df["corr"].mean()),
        "PATH_macro_mse": float(df["mse"].mean()),
        "PATH_macro_sign_acc": float(df["sign_acc"].mean()),
        "N": int(df["N"].iloc[0]) if len(df) else 0
    }
    return macro, df

def summarize_all_metrics(model, loader_idx, keys, topk=50):
    gene = collect_gene_arrays(model, loader_idx, topk=topk)
    path = collect_pathway_arrays(model, loader_idx, keys)
    macro, df_path = summarize_pathway_metrics(path)

    out = {
        "ALL978_mean_r": float(np.mean(gene["r_all"])),
        "ALL978_median_r": float(np.median(gene["r_all"])),
        f"Top{topk}_mean_r": float(np.mean(gene[f"r_top{topk}"])),
        f"Top{topk}_median_r": float(np.median(gene[f"r_top{topk}"])),
        **macro
    }
    return out, df_path, gene, path


In [ ]:
# ===== 17) EXP3a main: Hybrid fine-tune (start from MSE-only best) with lambda sweep =====
import time, math
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from tqdm.auto import tqdm

assert "best_mse" in globals() and best_mse.get("path") is not None, "Need best_mse['path'] from earlier training."
assert "OUT_DIR" in globals()

def load_ckpt_model(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = make_model()
    m.load_state_dict(ckpt["model"], strict=True)
    return m.to(device), ckpt

def pathway_loss_mse(pred_mat: torch.Tensor, true_mat: torch.Tensor) -> torch.Tensor:
    return ((pred_mat - true_mat) ** 2).mean()

def pathway_loss_corr(pred_mat: torch.Tensor, true_mat: torch.Tensor, eps=1e-8) -> torch.Tensor:
    # maximize correlation across samples within batch, averaged over pathways
    # pred/true: [B,P]
    p = pred_mat - pred_mat.mean(dim=0, keepdim=True)
    t = true_mat - true_mat.mean(dim=0, keepdim=True)
    num = (p * t).sum(dim=0)
    den = torch.sqrt((p*p).sum(dim=0) + eps) * torch.sqrt((t*t).sum(dim=0) + eps)
    corr = num / (den + eps)  # [P]
    return (1.0 - corr.mean())

def compute_pred_dnes_mat(pred_o: torch.Tensor, keys):
    cols = []
    for key in keys:
        up_mask = PATH_MASKS[key]["up_mask"].to(pred_o.device)
        dn_mask = PATH_MASKS[key]["dn_mask"].to(pred_o.device)
        rs = pathway_rs(pred_o.float(), up_mask, dn_mask)  # [B]
        dnes = to_dnes_key(rs, key)  # [B]
        cols.append(dnes)
    return torch.stack(cols, dim=1)  # [B,P]

# ---- baseline (for constraint on gene metric) on VAL ----
m0, _ = load_ckpt_model(best_mse["path"])
val0, _, _, _ = summarize_all_metrics(m0, val_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)
BASE_VAL_ALL = float(val0["ALL978_mean_r"])
print("[exp3a] baseline VAL (MSE-only best) ALL978_mean_r =", BASE_VAL_ALL)

# ---- hyperparams for hybrid fine-tune ----
EPOCHS_HYBRID = int(os.environ.get("EPOCHS_HYBRID", str(EPOCHS_HYBRID if "EPOCHS_HYBRID" in globals() else 4)))
LR_HYBRID = 3e-4
WEIGHT_DECAY_H = WEIGHT_DECAY
GRAD_CLIP_H = GRAD_CLIP

# lambda sweep (small -> moderate)
LAMBDA_LIST = [0.0, 0.01, 0.03, 0.10, 0.30]
# keep gene-level accuracy from collapsing: allow at most DROP_TOL
DROP_TOL = 0.02

# warmup to avoid sudden lambda shock
WARMUP_STEPS = 200

# choose pathway loss type
PATH_LOSS_TYPE = "mse"  # "mse" or "corr"
print("[exp3a] PATH_LOSS_TYPE =", PATH_LOSS_TYPE)

def train_hybrid_one(lam: float, out_tag: str):
    # start from MSE-only best
    model, _ = load_ckpt_model(best_mse["path"])
    model.train()

    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=LR_HYBRID, weight_decay=WEIGHT_DECAY_H)

    best = {"score": -1e18, "path": None, "epoch": -1, "lam": lam}
    hist = []
    step = 0

    for ep in range(1, EPOCHS_HYBRID + 1):
        model.train()
        losses = []
        t0 = time.time()

        for batch in tqdm(train_loader_idx, desc=f"hybrid(lam={lam}) e{ep}", leave=False):
            step += 1
            idx = batch["idx"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            fp = batch["fp"]
            fp = fp.to(device) if fp is not None else None
            y_z = batch["y_z"].to(device)

            pred_z = model(input_ids, attention_mask, fp=fp)
            mse = ((pred_z - y_z) ** 2).mean()  # scalar

            pred_o = inverse_standardize(pred_z).float()  # [B,G] original space

            # true dNES from cache: [B,P]
            true_cols = [torch.from_numpy(DNES_TRUE_ALL[k][idx.detach().cpu().numpy()]).to(device).float() for k in PATHWAY_KEYS_EXP3A]
            true_mat = torch.stack(true_cols, dim=1)  # [B,P]

            pred_mat = compute_pred_dnes_mat(pred_o, PATHWAY_KEYS_EXP3A)  # [B,P]

            if PATH_LOSS_TYPE == "mse":
                ploss = pathway_loss_mse(pred_mat, true_mat)
            else:
                ploss = pathway_loss_corr(pred_mat, true_mat)

            lam_eff = float(lam) * min(1.0, step / max(1, WARMUP_STEPS))
            loss = mse + lam_eff * ploss

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP_H)
            opt.step()

            losses.append(float(loss.detach().cpu().item()))

        # ---- validation ----
        model.eval()
        val_sum, df_val_path, _, _ = summarize_all_metrics(model, val_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)

        # constraint: do not drop too much on gene-level
        ok = (float(val_sum["ALL978_mean_r"]) >= BASE_VAL_ALL - DROP_TOL)

        # score: prioritize pathway macro corr, lightly penalize macro mse
        score = float(val_sum["PATH_macro_corr"] - 0.05 * val_sum["PATH_macro_mse"])
        if not ok:
            # hard penalty if gene metric collapses
            score = score - 10.0 * (BASE_VAL_ALL - float(val_sum["ALL978_mean_r"]) - DROP_TOL)

        rec = {
            "epoch": ep,
            "lam": lam,
            "train_loss": float(np.mean(losses)),
            "sec": time.time() - t0,
            "ok_gene": bool(ok),
            "score": float(score),
            **{f"val_{k}": float(v) for k, v in val_sum.items()},
        }
        hist.append(rec)

        print(f"[hybrid lam={lam}] ep={ep} train_loss={rec['train_loss']:.4f} | "
              f"VAL ALL978={rec['val_ALL978_mean_r']:.4f} Top50={rec['val_Top50_mean_r']:.4f} | "
              f"VAL PATH_macro_corr={rec['val_PATH_macro_corr']:.4f} sign={rec['val_PATH_macro_sign_acc']:.3f} | "
              f"ok_gene={ok} score={score:.4f}")

        # save best
        if score > best["score"]:
            best["score"] = score
            best["epoch"] = ep
            ckpt_path = OUT_DIR / f"{out_tag}_best.pt"
            torch.save({
                "model": model.state_dict(),
                "mode": "hybrid",
                "lam": lam,
                "epoch": ep,
                "score": float(score),
                "PATHWAY_KEYS": PATHWAY_KEYS_EXP3A,
                "DNES_SCALES": DNES_SCALES,
                "PATH_LOSS_TYPE": PATH_LOSS_TYPE,
                "BASE_VAL_ALL": BASE_VAL_ALL,
                "DROP_TOL": DROP_TOL,
            }, ckpt_path)
            best["path"] = str(ckpt_path)

    return best, pd.DataFrame(hist)

# ---- run sweep ----
hybrid_bests = []
hybrid_hists = {}

for lam in LAMBDA_LIST:
    tag = f"exp3a_hybrid_lam{str(lam).replace('.','p')}"
    b, h = train_hybrid_one(lam, out_tag=tag)
    hybrid_bests.append(b)
    hybrid_hists[lam] = h

df_best = pd.DataFrame(hybrid_bests).sort_values("score", ascending=False)
display(df_best)

best_hybrid = df_best.iloc[0].to_dict()
print("[exp3a] best_hybrid =", best_hybrid)


In [ ]:
# ===== 18) EXP3a paper table: TEST comparison (MSE-only vs DGSEA-only vs Hybrid-best) =====
import pandas as pd

assert "best_dg" in globals() and best_dg.get("path") is not None

m_mse, _ = load_ckpt_model(best_mse["path"])
m_dg,  _ = load_ckpt_model(best_dg["path"])
m_hy,  _ = load_ckpt_model(best_hybrid["path"])

test_mse_sum, test_mse_path_df, _, _ = summarize_all_metrics(m_mse, test_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)
test_dg_sum,  test_dg_path_df,  _, _ = summarize_all_metrics(m_dg,  test_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)
test_hy_sum,  test_hy_path_df,  _, _ = summarize_all_metrics(m_hy,  test_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)

summary3 = pd.DataFrame([
    {"setting": "MSE-only",   **test_mse_sum},
    {"setting": "DGSEA-only", **test_dg_sum},
    {"setting": "Hybrid(best)", **test_hy_sum},
]).set_index("setting")

display(summary3)

print("\n===== DELTA (Hybrid(best) - MSE-only) =====")
for k in ["PATH_macro_corr","PATH_macro_mse","PATH_macro_sign_acc","ALL978_mean_r","Top50_mean_r"]:
    dv = float(summary3.loc["Hybrid(best)", k] - summary3.loc["MSE-only", k])
    print(f"{k}: {dv:+.6f}")

# per-pathway tables
per_path = pd.concat([
    test_mse_path_df.add_prefix("MSE_"),
    test_dg_path_df.add_prefix("DGSEA_"),
    test_hy_path_df.add_prefix("HYB_"),
], axis=1)
display(per_path)


In [ ]:
# ===== release export) save core/per-path tables for consistency checks =====
TAB_DIR = OUT_DIR / "exp3a_outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

core_cols = ["ALL978_mean_r", "Top50_mean_r", "PATH_macro_corr", "PATH_macro_mse", "PATH_macro_sign_acc"]
core_table = summary3[core_cols].reset_index().rename(columns={"setting": "model"})
core_table.to_csv(TAB_DIR / "exp3a_table_test_models_core.csv", index=False)

per_path.to_csv(TAB_DIR / "exp3a_table_test_per_pathway.csv")

_PUB_DEFAULT = PROJECT_ROOT / "results" / "training_repro"
PUB_DIR = Path(os.environ.get("PUB_RESULTS_DIR", str(_PUB_DEFAULT))).expanduser().resolve()
PUB_DIR.mkdir(parents=True, exist_ok=True)
core_table.to_csv(PUB_DIR / "exp3a_table_test_models_core.csv", index=False)
per_path.to_csv(PUB_DIR / "exp3a_table_test_per_pathway.csv")

print("[release] saved:", TAB_DIR / "exp3a_table_test_models_core.csv")
print("[release] saved:", PUB_DIR / "exp3a_table_test_models_core.csv")


In [ ]:
# ===== 19) EXP3a: Paired bootstrap CI on TEST (Hybrid vs MSE) [fixed formatting + direction-aware probs] =====
import numpy as np

BOOT = 2000
SEED_BOOT = 7
rng = np.random.default_rng(SEED_BOOT)

# collect arrays for MSE and Hybrid
_, _, gene_mse, path_mse = summarize_all_metrics(m_mse, test_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)
_, _, gene_hy,  path_hy  = summarize_all_metrics(m_hy,  test_loader_idx, PATHWAY_KEYS_EXP3A, topk=50)

r_all_mse = gene_mse["r_all"]
r_all_hy  = gene_hy["r_all"]
r_top_mse = gene_mse["r_top50"]
r_top_hy  = gene_hy["r_top50"]

N = r_all_mse.shape[0]
assert r_all_hy.shape[0] == N

# stack pathway arrays into [N,P]
keys = PATHWAY_KEYS_EXP3A
P = len(keys)

pm = np.stack([path_mse[k]["pred"] for k in keys], axis=1)
tm = np.stack([path_mse[k]["true"] for k in keys], axis=1)
ph = np.stack([path_hy[k]["pred"]  for k in keys], axis=1)
th = np.stack([path_hy[k]["true"]  for k in keys], axis=1)
assert pm.shape == (N, P)

def pearson_1d(a, b, eps=1e-12):
    a = a - a.mean()
    b = b - b.mean()
    den = (np.sqrt((a*a).sum()) * np.sqrt((b*b).sum()) + eps)
    return float((a*b).sum() / den)

def metric_macro_corr(pred, true):
    return float(np.mean([pearson_1d(pred[:, j], true[:, j]) for j in range(pred.shape[1])]))

def metric_macro_mse(pred, true):
    return float(np.mean((pred - true) ** 2))

def metric_macro_sign(pred, true):
    return float(np.mean((pred >= 0) == (true >= 0)))

# observed deltas (Hybrid - MSE)
obs = {
    "ALL978_mean_r": float(r_all_hy.mean() - r_all_mse.mean()),         # higher better
    "Top50_mean_r":  float(r_top_hy.mean() - r_top_mse.mean()),         # higher better
    "PATH_macro_corr": float(metric_macro_corr(ph, th) - metric_macro_corr(pm, tm)),  # higher better
    "PATH_macro_mse":  float(metric_macro_mse(ph, th) - metric_macro_mse(pm, tm)),    # lower better (negative is good)
    "PATH_macro_sign_acc": float(metric_macro_sign(ph, th) - metric_macro_sign(pm, tm)),  # higher better
}
print("[obs delta] Hybrid - MSE:", obs)

# bootstrap deltas
deltas = {k: [] for k in obs.keys()}

for _ in range(BOOT):
    idx = rng.integers(0, N, size=N)  # paired resample
    deltas["ALL978_mean_r"].append(float(r_all_hy[idx].mean() - r_all_mse[idx].mean()))
    deltas["Top50_mean_r"].append(float(r_top_hy[idx].mean() - r_top_mse[idx].mean()))
    deltas["PATH_macro_corr"].append(float(metric_macro_corr(ph[idx], th[idx]) - metric_macro_corr(pm[idx], tm[idx])))
    deltas["PATH_macro_mse"].append(float(metric_macro_mse(ph[idx], th[idx]) - metric_macro_mse(pm[idx], tm[idx])))
    deltas["PATH_macro_sign_acc"].append(float(metric_macro_sign(ph[idx], th[idx]) - metric_macro_sign(pm[idx], tm[idx])))

def ci_and_prob(x, direction="gt"):
    """
    direction:
      - "gt": report P(Δ>0)
      - "lt": report P(Δ<0)
    """
    x = np.asarray(x, dtype=np.float64)
    lo, hi = np.quantile(x, [0.025, 0.975])
    if direction == "gt":
        p = float((x > 0).mean())
    else:
        p = float((x < 0).mean())
    return float(lo), float(hi), p

# which metrics are higher-better vs lower-better
HIGHER_BETTER = {"ALL978_mean_r", "Top50_mean_r", "PATH_macro_corr", "PATH_macro_sign_acc"}
LOWER_BETTER  = {"PATH_macro_mse"}

print("\n===== Paired bootstrap 95% CI for Δ(Hybrid - MSE) =====")
for k in obs.keys():
    if k in LOWER_BETTER:
        lo, hi, pgood = ci_and_prob(deltas[k], direction="lt")  # negative is improvement
        print(f"{k:>18s}: obs={obs[k]:+.6f} | CI=[{lo:+.6f}, {hi:+.6f}] | P(Δ<0)={pgood:.3f}")
    else:
        lo, hi, pgood = ci_and_prob(deltas[k], direction="gt")
        print(f"{k:>18s}: obs={obs[k]:+.6f} | CI=[{lo:+.6f}, {hi:+.6f}] | P(Δ>0)={pgood:.3f}")
